In [1]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv("https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv")
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [3]:
df.drop(["id", "Unnamed: 32"], axis=1, inplace=True)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:,0], test_size=0.2)

In [5]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [6]:
label = LabelEncoder()
y_train = label.fit_transform(y_train)
y_test = label.transform(y_test)

In [7]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

In [8]:
print(X_train_tensor.shape)
print(X_train_tensor.dtype)

torch.Size([455, 30])
torch.float32


- `__init__()` --> defines how and where the data is loaded.   

- `__len__()` --> returns the total number of rows/samples in the dataset.

- `__getitem__(index)` --> fetches and returns the specific data row and label at the given index.   

In [9]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [10]:
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

DataLoader - under the hood:   

1. Shuffle: at the start of an epoch, it uses a sampler to shuffle all data indices.   
2. Chunk: divides the shuffled indices into chunks based on the batch_size.   
3. Fetch: sends these indices to the Dataset's __getitem__ method to pull the actual data rows.   
4. Collate: combines the individual fetched rows into a single batch tensor using collate_fn and sends it to the training loop.  

In [11]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [12]:
# Defining the model

class TorchNN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features, 3),
            nn.ReLU(),
            nn.Linear(3, 1),
            nn.Sigmoid()
        )

    def forward(self, features):
        out = self.network(features)
        return out
    

In [13]:
learning_rate = 0.1
epochs = 25

In [14]:
# defining loss function
loss_function = nn.BCELoss()

In [15]:
# create Model
model = TorchNN(X_train_tensor.shape[1])

In [16]:
# defining optimizer
optimizer = torch.optim.SGD(model.parameters(), learning_rate)

# model.parameters() returns an iterator over all the trainable parameters in the model

In [17]:
# batch_size = 32
# epochs = 25
# n_samples = len(X_train_tensor)

# for epoch in range(epochs):
#     for start_idx in range(0, n_samples, batch_size):
#         end_idx = start_idx + batch_size

#         X_batch = X_train_tensor[start_idx:end_idx]
#         y_batch = y_train_tensor[start_idx:end_idx]

#         y_pred = model(X_batch)
#         loss = loss_function(y_pred, y_batch.reshape(-1,1))

#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

#     print(f"Epoch: {epoch} - Loss: {loss}")

In [18]:
# define loop
for epoch in range(epochs):
    for batch_features, batch_labels in train_loader:
        y_pred = model(batch_features)
        
        loss = loss_function(y_pred, batch_labels.reshape(-1,1))

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

    print(f"Epoch: {epoch} - Loss: {loss}")
     

Epoch: 0 - Loss: 0.6354231834411621
Epoch: 1 - Loss: 0.6709696650505066
Epoch: 2 - Loss: 0.4311957359313965
Epoch: 3 - Loss: 0.6482512354850769
Epoch: 4 - Loss: 0.4568159580230713
Epoch: 5 - Loss: 0.1376398503780365
Epoch: 6 - Loss: 0.1670096069574356
Epoch: 7 - Loss: 0.12987123429775238
Epoch: 8 - Loss: 0.11453700065612793
Epoch: 9 - Loss: 0.1091102734208107
Epoch: 10 - Loss: 0.07396329939365387
Epoch: 11 - Loss: 0.21806631982326508
Epoch: 12 - Loss: 0.11798029392957687
Epoch: 13 - Loss: 0.3717460632324219
Epoch: 14 - Loss: 0.4584314227104187
Epoch: 15 - Loss: 0.41945594549179077
Epoch: 16 - Loss: 0.058013152331113815
Epoch: 17 - Loss: 0.11668246984481812
Epoch: 18 - Loss: 0.06208037957549095
Epoch: 19 - Loss: 0.15357887744903564
Epoch: 20 - Loss: 0.07757794857025146
Epoch: 21 - Loss: 0.34036996960639954
Epoch: 22 - Loss: 0.10282031446695328
Epoch: 23 - Loss: 0.08944405615329742
Epoch: 24 - Loss: 0.04220382124185562


In [19]:
# evaluation
model.eval()
accuracy_list = []

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        y_pred = model(X_test_tensor)
        y_pred = (y_pred > 0.5).float()
        accuracy_list.append((y_pred == y_test_tensor.reshape(-1, 1)).float().mean())


accuracy = sum(accuracy_list)/len(accuracy_list)
print(f"Accuracy: {accuracy.item()*100:.2f}%")


Accuracy: 98.25%


* **Transformations:** Modifications applied to a single piece of data inside the dataset's `__getitem__` method before it gets batched. It handles preprocessing tasks like resizing an image, converting color to black-and-white, or making text lowercase.


* **Sampler:** The logic that dictates the exact order in which data indices are picked from the dataset. It decides whether to pull rows sequentially, shuffle them randomly, or use a custom strategy (like forcing representation for imbalanced classes).


* **Collate Function (`collate_fn`):** The final mechanism that dictates how the individually fetched rows are stitched together into one single batch tensor. Example, a custom collate function is necessary to pad variable-length data (like text sentences) with zeros so they can fit into the same batch.

Important `DataLoader` parameters:

- dataset: dataset object to pull data from (mandatory).   
- batch_size: number of samples per batch (default is 1).   
- shuffle: `True` to randomize the dataset indices every epoch.   
- num_workers: number of parallel CPU processes used to load data; speeds up I/O bottlenecks.   
- pin_memory: `True` to copy tensors into page-locked memory; speeds up data transfer to the GPU.   
- drop_last: `True` to discard the final incomplete batch if your total samples aren't perfectly divisible by the batch size; eg. preventing errors when using Batch Normalization, which expects consistent batch sizes.
- collate_fn: used to pass in custom batch-merging logic.   
- sampler: used to pass in custom sampling logic.   